# Phase 3: CodeT5+ 220M Full Fine-Tune

**Project:** Automated Code Documentation and Bug Report Analysis Using Fine-Tuned CodeT5+ Models

This notebook fine-tunes all parameters of `Salesforce/codet5p-220m` on the Phase 1 multi-language dataset using the Phase 2 pipeline conventions. It logs the efficiency numbers needed for the Phase 4 (LoRA) comparison and saves predictions for Phase 5 error analysis.

**Before running:** the Config cell is preset to the Weeks 1-4 outputs (`/content/drive/MyDrive/capstone/data/splits_v1`). Confirm the paths, then run top to bottom. Everything else is driven by that cell.

**Sections**
1. Setup and Google Drive
2. Config (frozen for Phase 4)
3. Compatibility helpers
4. Data loading and tokenization
5. Model and efficiency logging
6. Training (resumable)
7. Loss curves
8. Generation and evaluation (per language)
9. Save results


## 1. Setup and Google Drive

In [ ]:
!pip install -q transformers datasets accelerate evaluate sacrebleu rouge_score nltk sentencepiece

from google.colab import drive
drive.mount('/content/drive')

import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch runtime to A100)')


## 2. Config

These values are the ones to **freeze** for Phase 4. For the LoRA run, keep every setting identical except the ones you deliberately change (learning rate and the trainable parameters), and record why.

In [ ]:
import os, json, random, time
import numpy as np
import torch

# ---------- Paths ----------
BASE_DIR      = '/content/drive/MyDrive/capstone'
DATA_DIR      = f'{BASE_DIR}/data'
SPLIT_DIR     = f'{DATA_DIR}/splits_v2'                  # locked split from the Weeks 1-4 notebook: train.parquet, val.parquet, test.parquet
OUTPUT_DIR    = f'{BASE_DIR}/runs/codet5p_220m_full_ft_v2'
RESULTS_DIR   = f'{BASE_DIR}/results/phase3_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------- Dataset schema (matches the Weeks 1-4 notebook) ----------
# Columns in the parquet splits: input_text, target_text, source, language, task_type
SOURCE_COL  = 'input_text'
TARGET_COL  = 'target_text'

# The Weeks 1-4 dataset has no deterministic Logic suffix (targets are raw docstrings and bug summaries).
# Leave as None to score once on the references as stored. If you add a suffix later, set its marker string
# here and the evaluation adds a second set of scores with the suffix stripped.
LOGIC_SUFFIX_MARKER = None

# ---------- Model ----------
MODEL_NAME      = 'Salesforce/codet5p-220m'
MAX_SOURCE_LEN  = 512
MAX_TARGET_LEN  = 128

# ---------- Training hyperparameters (freeze for Phase 4 comparison) ----------
SEED                 = 42
LEARNING_RATE        = 5e-5
NUM_EPOCHS           = 3
PER_DEVICE_TRAIN_BS  = 16
PER_DEVICE_EVAL_BS   = 32
GRAD_ACCUM_STEPS     = 2          # effective batch = 16 * 2 = 32
WARMUP_RATIO         = 0.05
WEIGHT_DECAY         = 0.01
LR_SCHEDULER         = 'linear'
USE_BF16             = True

# ---------- Generation (identical for all three models) ----------
NUM_BEAMS       = 4
MAX_NEW_TOKENS  = 128
GEN_BATCH_SIZE  = 32

# Optional: set to an int to smoke-test on a small slice before the full run
DEBUG_SUBSET = None   # e.g. 2000

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)


## 3. Compatibility helpers

These are the two carry-over fixes: a safe tokenizer loader for CodeT5's RoBERTa-based tokenizer, and the `eval_strategy` argument name (used in the training arguments below).

In [ ]:
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

def load_tokenizer_safe(model_name, cache_root='/content/model_cache'):
    # Same fix as the Weeks 1-4 notebook: CodeT5+ 220M's added_tokens.json trips a strict type check
    # in newer transformers/tokenizers versions, so download a local copy, drop that file, and
    # normalize special_tokens_map.json before loading.
    local_dir = os.path.join(cache_root, model_name.replace('/', '__'))
    snapshot_download(repo_id=model_name, local_dir=local_dir)

    added_tokens_path = os.path.join(local_dir, 'added_tokens.json')
    if os.path.exists(added_tokens_path):
        os.remove(added_tokens_path)

    special_tokens_path = os.path.join(local_dir, 'special_tokens_map.json')
    if os.path.exists(special_tokens_path):
        with open(special_tokens_path, 'r', encoding='utf-8') as f:
            special_tokens = json.load(f)
        additional = special_tokens.get('additional_special_tokens')
        if additional and isinstance(additional[0], dict):
            special_tokens['additional_special_tokens'] = [
                t['content'] if isinstance(t, dict) else t for t in additional
            ]
            with open(special_tokens_path, 'w', encoding='utf-8') as f:
                json.dump(special_tokens, f)

    return AutoTokenizer.from_pretrained(local_dir)

tokenizer = load_tokenizer_safe(MODEL_NAME)
print('Tokenizer:', type(tokenizer).__name__, '| vocab size:', len(tokenizer))


## 4. Data loading and tokenization

In [ ]:
import pandas as pd
from datasets import Dataset

def load_split(name):
    return Dataset.from_pandas(pd.read_parquet(f'{SPLIT_DIR}/{name}.parquet'), preserve_index=False)

train_raw, val_raw, test_raw = load_split('train'), load_split('val'), load_split('test')

if DEBUG_SUBSET:
    train_raw = train_raw.select(range(min(DEBUG_SUBSET, len(train_raw))))
    val_raw   = val_raw.select(range(min(DEBUG_SUBSET, len(val_raw))))
    test_raw  = test_raw.select(range(min(DEBUG_SUBSET, len(test_raw))))
    print('DEBUG subset in use')

print(f'Train: {len(train_raw):,} | Val: {len(val_raw):,} | Test: {len(test_raw):,}')
print('Columns:', train_raw.column_names)
print(test_raw.to_pandas().groupby(['source', 'language']).size())


In [ ]:
def preprocess(batch):
    model_inputs = tokenizer(batch[SOURCE_COL], max_length=MAX_SOURCE_LEN, truncation=True)
    labels = tokenizer(text_target=batch[TARGET_COL], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tok_train = train_raw.map(preprocess, batched=True, remove_columns=train_raw.column_names, desc='Tokenizing train')
tok_val   = val_raw.map(preprocess,   batched=True, remove_columns=val_raw.column_names,   desc='Tokenizing validation')

# Truncation check: share of examples hitting the length caps (useful to report)
src_len = np.array([len(x) for x in tok_train['input_ids']])
tgt_len = np.array([len(x) for x in tok_train['labels']])
print(f'Source truncated: {(src_len >= MAX_SOURCE_LEN).mean():.2%} | Target truncated: {(tgt_len >= MAX_TARGET_LEN).mean():.2%}')
print(f'Median source len: {int(np.median(src_len))} | Median target len: {int(np.median(tgt_len))}')


## 5. Model and efficiency logging

In [ ]:
from transformers import AutoModelForSeq2SeqLM, TrainerCallback
import time

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, dtype=torch.float32)   # newer Transformers loads in the checkpoint dtype (fp16) by default
print('Weights dtype:', next(model.parameters()).dtype)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,} ({trainable_params/total_params:.2%})')

class EfficiencyCallback(TrainerCallback):
    """Records wall-clock time per epoch and peak GPU memory, for the Phase 4/5 comparison."""
    def __init__(self):
        self.epoch_times, self.epoch_peak_mem_gb = [], []
        self._t0 = None
        self.train_start = None
    def on_train_begin(self, args, state, control, **kw):
        self.train_start = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    def on_epoch_begin(self, args, state, control, **kw):
        self._t0 = time.time()
    def on_epoch_end(self, args, state, control, **kw):
        self.epoch_times.append(time.time() - self._t0)
        if torch.cuda.is_available():
            self.epoch_peak_mem_gb.append(torch.cuda.max_memory_allocated() / 1e9)

eff_cb = EfficiencyCallback()


## 6. Training (resumable)

Training evaluates on **validation loss** each epoch and keeps the best checkpoint by `eval_loss`. Generation metrics (BLEU-4, ROUGE-L, METEOR) are computed after training in Section 8, which keeps each epoch fast. Checkpoints go to Google Drive, and the cell resumes automatically if the Colab session drops.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint

collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)

import math
steps_per_epoch = math.ceil(len(tok_train) / (PER_DEVICE_TRAIN_BS * GRAD_ACCUM_STEPS))
WARMUP_STEPS = int(WARMUP_RATIO * steps_per_epoch * NUM_EPOCHS)   # warmup_ratio was removed in newer Transformers
print(f'Steps/epoch: {steps_per_epoch:,} | total: {steps_per_epoch * NUM_EPOCHS:,} | warmup steps: {WARMUP_STEPS:,}')

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BS,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    bf16=USE_BF16 and torch.cuda.is_available(),
    eval_strategy='epoch',            # renamed from evaluation_strategy in newer Transformers
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_strategy='steps',
    logging_steps=100,
    predict_with_generate=False,
    dataloader_num_workers=2,
    report_to='none',
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=collator,
    processing_class=tokenizer,       # use tokenizer=tokenizer on Transformers < 4.46
    callbacks=[eff_cb],
)

last_ckpt = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
print('Resuming from:', last_ckpt if last_ckpt else 'scratch')

train_result = trainer.train(resume_from_checkpoint=last_ckpt)
print(train_result.metrics)


In [ ]:
BEST_DIR = f'{OUTPUT_DIR}/best'
trainer.save_model(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)
print('Saved best model to', BEST_DIR)
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best eval_loss:', trainer.state.best_metric)


## 7. Loss curves

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_steps  = [l['step'] for l in log if 'loss' in l and 'eval_loss' not in l]
train_losses = [l['loss'] for l in log if 'loss' in l and 'eval_loss' not in l]
eval_steps   = [l['step'] for l in log if 'eval_loss' in l]
eval_losses  = [l['eval_loss'] for l in log if 'eval_loss' in l]

plt.figure(figsize=(8, 4.5))
plt.plot(train_steps, train_losses, label='Train loss', alpha=0.8)
plt.plot(eval_steps, eval_losses, marker='o', label='Validation loss')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('CodeT5+ 220M full fine-tune: loss curves')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_curves_full_ft.png', dpi=200)
plt.show()


## 8. Generation and evaluation

Settings are fixed in the Config cell so all three models are scored identically. Results are reported per language (the six CodeSearchNet languages plus GitBugs) and combined.

The Weeks 1-4 dataset uses raw docstrings and bug summaries as targets, with no deterministic Logic suffix, so scores are computed once on the references as stored. If `LOGIC_SUFFIX_MARKER` is set in the Config cell, a second set of scores with the suffix stripped is added automatically.

In [ ]:
import evaluate, pandas as pd, re

bleu_m   = evaluate.load('bleu')
rouge_m  = evaluate.load('rouge')
meteor_m = evaluate.load('meteor')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = trainer.model.to(device).eval()

@torch.no_grad()
def generate_all(texts):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))   # length-sorted batching for speed
    outputs = [None] * len(texts)
    for start in range(0, len(order), GEN_BATCH_SIZE):
        idx = order[start:start + GEN_BATCH_SIZE]
        batch = tokenizer([texts[i] for i in idx], max_length=MAX_SOURCE_LEN, truncation=True,
                          padding=True, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=(device == 'cuda' and USE_BF16)):
            gen = model.generate(**batch, num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS, early_stopping=True)
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        for i, d in zip(idx, decoded):
            outputs[i] = d.strip()
    return outputs

t0 = time.time()
test_sources = test_raw[SOURCE_COL]
test_refs    = test_raw[TARGET_COL]
test_preds   = generate_all(test_sources)
gen_seconds  = time.time() - t0
print(f'Generated {len(test_preds):,} predictions in {gen_seconds/60:.1f} min '
      f'({1000*gen_seconds/len(test_preds):.1f} ms/example)')


In [ ]:
def strip_logic(text):
    pos = text.find(LOGIC_SUFFIX_MARKER)
    return (text[:pos] if pos != -1 else text).strip()

def score(preds, refs):
    preds = [p if p.strip() else ' ' for p in preds]
    refs  = [r if r.strip() else ' ' for r in refs]
    b = bleu_m.compute(predictions=preds, references=[[r] for r in refs], max_order=4)['bleu']
    r = rouge_m.compute(predictions=preds, references=refs)['rougeL']
    m = meteor_m.compute(predictions=preds, references=refs)['meteor']
    return {'BLEU-4': 100 * b, 'ROUGE-L': 100 * r, 'METEOR': 100 * m}

df = pd.DataFrame({'source': test_sources, 'reference': test_refs, 'prediction': test_preds})
df['group'] = [('gitbugs' if src == 'gitbugs' else lang) for src, lang in zip(test_raw['source'], test_raw['language'])]
VARIANTS = [('as_stored', 'prediction', 'reference')]
if LOGIC_SUFFIX_MARKER:
    df['reference_nosuffix']  = df['reference'].map(strip_logic)
    df['prediction_nosuffix'] = df['prediction'].map(strip_logic)
    VARIANTS.append(('without_suffix', 'prediction_nosuffix', 'reference_nosuffix'))
    has_marker = df['reference'].str.contains(re.escape(LOGIC_SUFFIX_MARKER)).mean()
    print(f'References containing the Logic marker: {has_marker:.1%} (if 0%, fix LOGIC_SUFFIX_MARKER in the Config cell)')
else:
    print('No Logic suffix configured: scoring once on references as stored.')

rows = []
def add_rows(label, sub):
    for variant, p, r in VARIANTS:
        s = score(sub[p].tolist(), sub[r].tolist())
        rows.append({'group': label, 'variant': variant, 'n': len(sub), **s})

add_rows('ALL', df)
for g, sub in df.groupby('group'):
    add_rows(g, sub)

results = pd.DataFrame(rows).round(2)
results


## 9. Save results

In [ ]:
summary = {
    'model': MODEL_NAME,
    'method': 'full_fine_tune',
    'hyperparameters': {
        'learning_rate': LEARNING_RATE, 'epochs': NUM_EPOCHS,
        'effective_batch_size': PER_DEVICE_TRAIN_BS * GRAD_ACCUM_STEPS,
        'warmup_ratio': WARMUP_RATIO, 'weight_decay': WEIGHT_DECAY,
        'max_source_len': MAX_SOURCE_LEN, 'max_target_len': MAX_TARGET_LEN,
        'bf16': USE_BF16, 'seed': SEED,
    },
    'generation': {'num_beams': NUM_BEAMS, 'max_new_tokens': MAX_NEW_TOKENS},
    'efficiency': {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'trainable_pct': round(100 * trainable_params / total_params, 2),
        'epoch_times_sec': [round(t, 1) for t in eff_cb.epoch_times],
        'total_train_time_min': round((time.time() - eff_cb.train_start) / 60, 1) if eff_cb.train_start else None,
        'peak_gpu_mem_gb': round(max(eff_cb.epoch_peak_mem_gb), 2) if eff_cb.epoch_peak_mem_gb else None,
        'generation_ms_per_example': round(1000 * gen_seconds / len(test_preds), 1),
    },
    'best_eval_loss': trainer.state.best_metric,
    'best_checkpoint': trainer.state.best_model_checkpoint,
}

results.to_csv(f'{RESULTS_DIR}/metrics_full_ft.csv', index=False)
df.to_csv(f'{RESULTS_DIR}/predictions_full_ft.csv', index=False)
with open(f'{RESULTS_DIR}/summary_full_ft.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print('\nSaved to', RESULTS_DIR)


### Notes for Phase 4 and Phase 5

- Reuse this notebook for the LoRA run: wrap the model with `peft.get_peft_model(...)`, raise the learning rate (LoRA commonly uses 1e-4 to 5e-4), and keep every other setting identical. Save results with a `_lora` suffix.
- `summary_full_ft.json` already has the fields needed for the efficiency comparison table (trainable parameters, time per epoch, peak memory, generation latency).
- `predictions_full_ft.csv` contains source, reference, and prediction (with and without the suffix) for error analysis.
- Spring scores were inflated by the deterministic Logic suffix. This dataset has no suffix, so expect much lower and more realistic scores than Spring, and state that comparison explicitly in the results section.
